## Linear Regression Model

In [1]:
import pandas as pd
import numpy as np
from configs.system import project_dir
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:
p_dir = project_dir()

dir_dataset = p_dir.FEATURED_DIR
dataset_path = dir_dataset/'featured_dataset.csv'
df = pd.read_csv(dataset_path,index_col=0)
df.sample(3)

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,...,Duration,Demand,StartMonth,StartWeek,EndMonth,EndWeek,DemandPerMonth,StartDayOfWeek,DemandPerDay,Shift
69811,346,1892,0,public parking lot,0,2022-09-03,17:38:44,2022-09-03,20:17:01,2022-09-03 17:38:00,...,158,9.03,9,5,9,5,111771.80,Saturday,17.41,Night
58300,757,956,0,public institution,0,2022-08-13,07:34:04,2022-08-13,07:40:51,2022-08-13 07:34:00,...,7,0.70,8,5,8,5,143041.27,Saturday,17.41,Day
44327,354,431,1,camping,0,2022-03-02,00:56:44,2022-03-02,05:24:13,2022-03-02 00:56:00,...,267,30.27,3,2,3,2,103896.05,Wednesday,17.53,Day


In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 72826 entries, 0 to 72855
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UserID          72826 non-null  int64  
 1   ChargerID       72826 non-null  int64  
 2   ChargerCompany  72826 non-null  int64  
 3   Location        72826 non-null  str    
 4   ChargerType     72826 non-null  int64  
 5   StartDay        72826 non-null  str    
 6   StartTime       72826 non-null  str    
 7   EndDay          72826 non-null  str    
 8   EndTime         72826 non-null  str    
 9   StartDatetime   72826 non-null  str    
 10  EndDatetime     72826 non-null  str    
 11  Duration        72826 non-null  int64  
 12  Demand          72826 non-null  float64
 13  StartMonth      72826 non-null  int64  
 14  StartWeek       72826 non-null  int64  
 15  EndMonth        72826 non-null  int64  
 16  EndWeek         72826 non-null  int64  
 17  DemandPerMonth  72826 non-null  float64
 18  St

In [4]:
X = df[['Location','StartDayOfWeek','ChargerCompany','ChargerType','Duration','StartMonth','StartWeek','DemandPerMonth','DemandPerDay','Shift']]
y = df['Demand']

In [5]:
categorical_columns = ['Location','StartDayOfWeek','Shift']
numerical_columns = ['ChargerCompany','ChargerType','Duration','StartMonth','StartWeek','DemandPerMonth','DemandPerDay']

In [6]:
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_columns),
        ('num',StandardScaler(),numerical_columns)
    ]
)

In [8]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

In [9]:
ln = LinearRegression()
ln.fit(X_train_prep,y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](30,)","[ 6.46,-1.89, 7.21,..., 0.02,-0.18, 0.02]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,18.74
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,30
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(25)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](30,)","[337.12,315.51,265.81,..., 0. , 0. , 0. ]"


In [10]:
y_pred = ln.predict(X_test_prep)


In [11]:
r2_ln = r2_score(y_test,y_pred)
mae_ln = mean_absolute_error(y_test,y_pred)
mse_ln = mean_squared_error(y_test,y_pred)

In [12]:
print(f'R2 score {r2_ln}')
print(f'Mean Absolute Error {mae_ln}')
print(f'Mean Squared Absolute Error: {mse_ln}')

R2 score 0.5681699963546796
Mean Absolute Error 5.618755143141776
Mean Squared Absolute Error: 77.81493921433258


In [13]:
results = []

results.append({
    'model' : 'Linear Regression',
    'R2 score' : r2_ln,
    'MAE' : mae_ln,
    'MSE' : mse_ln

})

In [14]:
results

[{'model': 'Linear Regression',
  'R2 score': 0.5681699963546796,
  'MAE': 5.618755143141776,
  'MSE': 77.81493921433258}]

### Decision Tree Regression

In [15]:
depths = [1,2,3,4,5,6,7,8,9,10]


dt_results = []

for depth in depths:
    dt = DecisionTreeRegressor(max_depth=depth)
    # dt = DecisionTreeRegressor(min_samples_split=30)

    dt.fit(X_train_prep,y_train)

    y_pred = dt.predict(X_test_prep)
    y_pred_train = dt.predict(X_train_prep)

    r2_score_dt = r2_score(y_train,y_pred_train)
    dt_results.append({
        'depth' : depth,
        'r2_score':r2_score_dt

    })
    

In [16]:
dt_results

[{'depth': 1, 'r2_score': 0.42493397525328835},
 {'depth': 2, 'r2_score': 0.5088495565112461},
 {'depth': 3, 'r2_score': 0.6024186459371077},
 {'depth': 4, 'r2_score': 0.7116415823447548},
 {'depth': 5, 'r2_score': 0.746433427127474},
 {'depth': 6, 'r2_score': 0.7663729032088853},
 {'depth': 7, 'r2_score': 0.781067793296446},
 {'depth': 8, 'r2_score': 0.7964437393546133},
 {'depth': 9, 'r2_score': 0.8076284321332893},
 {'depth': 10, 'r2_score': 0.8215151779764399}]

In [17]:
samples_split = [5,10,15,20,25,30,40,50]
d_r = []
for s in samples_split:
    dt = DecisionTreeRegressor(min_samples_split=s)
    dt.fit(X_train_prep,y_train)

    y_pred = dt.predict(X_test_prep)
    y_train_pred = dt.predict(X_train_prep)

    r2_score_train_d = r2_score(y_train,y_train_pred)
    r2_score_d = r2_score(y_test,y_pred)

    d_r.append({
        'sample_split' : s,
        'r2_split' : r2_score_d,
        'r2_score_train' : r2_score_train_d
    })



In [18]:
d_r

[{'sample_split': 5,
  'r2_split': 0.6622061092215472,
  'r2_score_train': 0.9407281373824069},
 {'sample_split': 10,
  'r2_split': 0.7131019415450129,
  'r2_score_train': 0.9004834552258779},
 {'sample_split': 15,
  'r2_split': 0.7307188056268541,
  'r2_score_train': 0.8785364992495757},
 {'sample_split': 20,
  'r2_split': 0.7474452297089058,
  'r2_score_train': 0.864446518990667},
 {'sample_split': 25,
  'r2_split': 0.7544069129767283,
  'r2_score_train': 0.8570240773094502},
 {'sample_split': 30,
  'r2_split': 0.7616891735453399,
  'r2_score_train': 0.8495635897261594},
 {'sample_split': 40,
  'r2_split': 0.7668578958653827,
  'r2_score_train': 0.8390111245256349},
 {'sample_split': 50,
  'r2_split': 0.7709093070384975,
  'r2_score_train': 0.8328392613694495}]

In [19]:
d_tree = DecisionTreeRegressor(min_samples_split=25)
d_tree.fit(X_train_prep,y_train)

y_pred = d_tree.predict(X_test_prep)
# y_pred_train = d_tree.predict(X_train_prep)

r2_score_d_tree = r2_score(y_test,y_pred)
mae_d_tree = mean_absolute_error(y_test,y_pred)
mse_d_tree = mean_squared_error(y_test,y_pred)

results.append({
    'model' : 'DecisionTreeRegressor',
    'R2 score' : r2_score_d_tree,
    'MAE' : mae_d_tree,
    'MSE' : mse_d_tree
})



In [20]:
results

[{'model': 'Linear Regression',
  'R2 score': 0.5681699963546796,
  'MAE': 5.618755143141776,
  'MSE': 77.81493921433258},
 {'model': 'DecisionTreeRegressor',
  'R2 score': 0.754321288937186,
  'MAE': 3.602868947048413,
  'MSE': 44.270832981096895}]

## Random Forest 

In [21]:
from sklearn.ensemble import RandomForestRegressor

estimaters = [10,100]
r_re = []
for n in estimaters:
    rf = RandomForestRegressor(n_estimators=n,n_jobs=-1,random_state=42)

    rf.fit(X_train_prep,y_train)

    y_pred_r = rf.predict(X_test_prep)
    y_pred_r_train = rf.predict(X_train_prep)

    r2_score_r = r2_score(y_test,y_pred_r)
    r2_score_r_train = r2_score(y_train, y_pred_r_train)

    r_re.append({
        'n_estimators' : n,
        'Train r2 ' : r2_score_r_train,
        'Test R2' : r2_score_r
    })

In [22]:
r_re

[{'n_estimators': 10,
  'Train r2 ': 0.9450607138891386,
  'Test R2': 0.758099686357649},
 {'n_estimators': 100,
  'Train r2 ': 0.95469472304676,
  'Test R2': 0.7763237097303186}]

In [23]:
depths = [1,2,4,5,7,8,9,10]

for depth in depths:
    rf_d = RandomForestRegressor(max_depth=depth,n_estimators=10,n_jobs=-1)

    rf_d.fit(X_train_prep,y_train)

    y_pred = rf_d.predict(X_test_prep)
    y_pred_train = rf_d.predict(X_train_prep)

    test_r2 = r2_score(y_test,y_pred)
    train_r2 = r2_score(y_train,y_pred_train)


    r_re.append({
        'depth' : depth,
        'Train r2' : train_r2,
        'Test R2' : test_r2
    })

In [24]:
r_re

[{'n_estimators': 10,
  'Train r2 ': 0.9450607138891386,
  'Test R2': 0.758099686357649},
 {'n_estimators': 100,
  'Train r2 ': 0.95469472304676,
  'Test R2': 0.7763237097303186},
 {'depth': 1, 'Train r2': 0.4306906848485633, 'Test R2': 0.41891659577152907},
 {'depth': 2, 'Train r2': 0.5133078353801246, 'Test R2': 0.5019738370518108},
 {'depth': 4, 'Train r2': 0.7169330275871333, 'Test R2': 0.7140486368320903},
 {'depth': 5, 'Train r2': 0.7516768263416823, 'Test R2': 0.7480511193052064},
 {'depth': 7, 'Train r2': 0.7873170664449795, 'Test R2': 0.7771255588911887},
 {'depth': 8, 'Train r2': 0.8025709754378499, 'Test R2': 0.7869782306025427},
 {'depth': 9, 'Train r2': 0.8167528472710766, 'Test R2': 0.7909081653151814},
 {'depth': 10, 'Train r2': 0.8291296496598684, 'Test R2': 0.789332796434851}]

In [25]:
X_train_prep[:1]

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  0.        ,  1.        ,  0.76951367, -0.50824732,
         0.55697578,  0.5382157 , -0.98739955,  1.50845844, -1.62488668]])

In [26]:
results

[{'model': 'Linear Regression',
  'R2 score': 0.5681699963546796,
  'MAE': 5.618755143141776,
  'MSE': 77.81493921433258},
 {'model': 'DecisionTreeRegressor',
  'R2 score': 0.754321288937186,
  'MAE': 3.602868947048413,
  'MSE': 44.270832981096895}]

In [27]:
rf_tree = RandomForestRegressor(max_depth=7,n_estimators=10,n_jobs=-1,random_state=42)
rf_tree.fit(X_train_prep,y_train)

pred_y = rf_tree.predict(X_test_prep)

r2_score_rf = r2_score(y_test,pred_y)
mae_rf = mean_absolute_error(y_test,pred_y)
mse_rf = mean_squared_error(y_test,pred_y)

results.append({
    'model' : 'RandomForestRegressor',
    'R2 score' : r2_score_rf,
    'MAE' : mae_rf,
    'MSE' : mse_rf
})

In [28]:
results

[{'model': 'Linear Regression',
  'R2 score': 0.5681699963546796,
  'MAE': 5.618755143141776,
  'MSE': 77.81493921433258},
 {'model': 'DecisionTreeRegressor',
  'R2 score': 0.754321288937186,
  'MAE': 3.602868947048413,
  'MSE': 44.270832981096895},
 {'model': 'RandomForestRegressor',
  'R2 score': 0.7777906629372556,
  'MAE': 3.5761078782247884,
  'MSE': 40.041696756662986}]

## Gradient Boosting Regressor

In [29]:
from sklearn.ensemble import GradientBoostingRegressor

rates = [0.01,0.1,1.0]
results_gb = []

for r in rates:
    gb = GradientBoostingRegressor(learning_rate=r,n_estimators=100,random_state=42,max_depth=6)
    gb.fit(X_train_prep,y_train)
    gb.predict(X_train_prep)

    y_pred = gb.predict(X_test_prep)
    y_pred_train = gb.predict(X_train_prep)

    r2_score_train = r2_score(y_train,y_pred_train)
    r2_score_test = r2_score(y_test,y_pred)

    # results_gb = []
    results_gb.append({
        'rates': r,
        'R2 train' : r2_score_train,
        'R2 Test' : r2_score_test
     })

In [30]:
results_gb

[{'rates': 0.01,
  'R2 train': 0.6720081735522998,
  'R2 Test': 0.6693467021185209},
 {'rates': 0.1, 'R2 train': 0.8286344144814752, 'R2 Test': 0.7962797202061224},
 {'rates': 1.0, 'R2 train': 0.8964757264625112, 'R2 Test': 0.7187040056489324}]

In [31]:
gbr = GradientBoostingRegressor(max_depth=5,n_estimators=100,learning_rate=0.1,random_state=42)
gbr.fit(X_train_prep,y_train)

y_pred_gbr = gbr.predict(X_test_prep)
y_pred_train_gbr = gbr.predict(X_train_prep)

r2_score_gb = r2_score(y_test,y_pred_gbr)
r2_score_train_gb = r2_score(y_train,y_pred_train_gbr)

mae_gb = mean_absolute_error(y_test,y_pred_gbr)
mse_gb = mean_squared_error(y_test,y_pred_gbr)



In [32]:
results

[{'model': 'Linear Regression',
  'R2 score': 0.5681699963546796,
  'MAE': 5.618755143141776,
  'MSE': 77.81493921433258},
 {'model': 'DecisionTreeRegressor',
  'R2 score': 0.754321288937186,
  'MAE': 3.602868947048413,
  'MSE': 44.270832981096895},
 {'model': 'RandomForestRegressor',
  'R2 score': 0.7777906629372556,
  'MAE': 3.5761078782247884,
  'MSE': 40.041696756662986}]

In [33]:
results.append({
    'model' : 'Gradient Boosting',
    'R2 score' : r2_score_gb,
    'MAE' : mae_gb,
    'MSE' : mse_gb
})

In [34]:
results

[{'model': 'Linear Regression',
  'R2 score': 0.5681699963546796,
  'MAE': 5.618755143141776,
  'MSE': 77.81493921433258},
 {'model': 'DecisionTreeRegressor',
  'R2 score': 0.754321288937186,
  'MAE': 3.602868947048413,
  'MSE': 44.270832981096895},
 {'model': 'RandomForestRegressor',
  'R2 score': 0.7777906629372556,
  'MAE': 3.5761078782247884,
  'MSE': 40.041696756662986},
 {'model': 'Gradient Boosting',
  'R2 score': 0.790323841378304,
  'MAE': 3.4755008551712523,
  'MSE': 37.78324201678901}]

### XGBoost Regressor

In [35]:
from xgboost import XGBRegressor

estimaters = [100,200,300,500]
xg_r = []

for n in estimaters:
    xg = XGBRegressor(n_estimators=n,max_depth=4,subsample=0.8, colsample_bytree=0.8,n_jobs=-1,learning_rate=0.1)

    xg.fit(X_train_prep,y_train)
    y_pred = xg.predict(X_test_prep)
    y_pred_train = xg.predict(X_train_prep)

    r2_xg = r2_score(y_test,y_pred)
    r2_xg_train = r2_score(y_train,y_pred_train)

    xg_r.append({
        'estimaters' : n,
        'r2_train' : r2_xg_train,
        'r2_test' : r2_xg
    })



In [36]:
xg_r

[{'estimaters': 100,
  'r2_train': 0.7940750289483822,
  'r2_test': 0.7848811256648953},
 {'estimaters': 200,
  'r2_train': 0.8078726494252725,
  'r2_test': 0.7911501808061563},
 {'estimaters': 300,
  'r2_train': 0.8151411886099589,
  'r2_test': 0.7929029978421294},
 {'estimaters': 500,
  'r2_train': 0.8232726712297667,
  'r2_test': 0.7940186254031475}]

In [37]:
xg_r

[{'estimaters': 100,
  'r2_train': 0.7940750289483822,
  'r2_test': 0.7848811256648953},
 {'estimaters': 200,
  'r2_train': 0.8078726494252725,
  'r2_test': 0.7911501808061563},
 {'estimaters': 300,
  'r2_train': 0.8151411886099589,
  'r2_test': 0.7929029978421294},
 {'estimaters': 500,
  'r2_train': 0.8232726712297667,
  'r2_test': 0.7940186254031475}]

In [38]:
depths = [4,5,6,7]
xg_depth = []
for depth in depths:
    xg = XGBRegressor(max_depth=depth,n_estimators=200,learning_rate=0.01,random_state=42,subsample=0.8,colsample_bytree=0.8,n_jobs=-1)

    xg.fit(X_train_prep,y_train)

    y_pred = xg.predict(X_test_prep)
    y_pred_train = xg.predict(X_train_prep)

    r2_test = r2_score(y_test,y_pred)
    r2_train = r2_score(y_train,y_pred_train)

    xg_depth.append({
        'depth' : depth,
        'r2_test' : r2_test,
        'r2_train' : r2_train
    })

    

In [39]:
xg_depth

[{'depth': 4, 'r2_test': 0.7144151771935847, 'r2_train': 0.7149209173244249},
 {'depth': 5, 'r2_test': 0.7282396359954513, 'r2_train': 0.7312746846137353},
 {'depth': 6, 'r2_test': 0.738757666657616, 'r2_train': 0.7438203570678912},
 {'depth': 7, 'r2_test': 0.7465363057130789, 'r2_train': 0.7559380457056502}]

In [40]:
xg_r = XGBRegressor(max_depth=6,n_estimators=200,learning_rate=0.1,random_state=42,n_jobs=-1)

xg_r.fit(X_train_prep,y_train)

y_pred = xg.predict(X_test_prep)
y_pred_train = xg.predict(X_train_prep)

r2_score_test = r2_score(y_test,y_pred)
r2_score_train = r2_score(y_train,y_pred_train)



In [41]:
mae_xg = mean_absolute_error(y_test,y_pred)
mse_xg = mean_squared_error(y_test,y_pred)

In [42]:
results.append({
    'model' : 'XGBRegressor',
    'R2 score' : r2_score_test,
    'MAE' : mae_xg,
    'MSE' : mse_xg
})

In [43]:
results

[{'model': 'Linear Regression',
  'R2 score': 0.5681699963546796,
  'MAE': 5.618755143141776,
  'MSE': 77.81493921433258},
 {'model': 'DecisionTreeRegressor',
  'R2 score': 0.754321288937186,
  'MAE': 3.602868947048413,
  'MSE': 44.270832981096895},
 {'model': 'RandomForestRegressor',
  'R2 score': 0.7777906629372556,
  'MAE': 3.5761078782247884,
  'MSE': 40.041696756662986},
 {'model': 'Gradient Boosting',
  'R2 score': 0.790323841378304,
  'MAE': 3.4755008551712523,
  'MSE': 37.78324201678901},
 {'model': 'XGBRegressor',
  'R2 score': 0.7465363057130789,
  'MAE': 4.534751806548544,
  'MSE': 45.67367204103875}]